[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Greedy Algorithms** {#greedy-algorithms}

A **greedy algorithm** constructs a solution through a sequence of commitments. At each step it chooses the feasible option that looks best according to a local rule, records that choice permanently, and solves the smaller problem that remains. The method resembles packing for a trip by repeatedly taking the most useful item that still fits. This is fast and decisive, but it is only correct when an early choice cannot block a better global solution.

Greedy design is therefore not synonymous with "take the largest" or "take the smallest." It is a three-part claim:

1. a precisely defined local rule identifies the next choice;
2. the choice is **safe**, meaning that at least one globally optimal solution contains it; and
3. after committing to it, the residual problem has the same useful structure.

This chapter develops that claim from several angles. Interval scheduling shows how sorting creates a simple feasibility frontier. Huffman coding shows how a priority queue repeatedly exposes the next safe pair. Minimum spanning trees use the cut property to certify safe edges. Counterexamples then show why an attractive local rule is not evidence of correctness.

::: {.callout-important}
A greedy algorithm normally explores one solution path. Its speed comes from discarding alternatives, so the correctness proof must justify exactly why those alternatives are unnecessary.
:::


### **Greedy Choice and Optimal Substructure** {#greedy-choice-and-optimal-substructure}

The **greedy-choice property** says that a globally optimal solution can be reached after making the algorithm's locally preferred first choice. It does not require every optimal solution to start this way. It is enough to prove that at least one optimal solution can be rearranged to contain the greedy choice without becoming worse.

**Optimal substructure** is a different property. After the first choice is fixed, the remaining part of an optimal solution must itself be optimal for the residual problem. If <code>P</code> is the original problem, <code>g</code> is a safe greedy choice, and <code>P_g</code> is what remains after choosing <code>g</code>, the structure can be written schematically as

$$
\operatorname{OPT}(P)=\operatorname{combine}\!\left(g,\operatorname{OPT}(P_g)\right).
$$

Here, $\operatorname{OPT}(P)$ denotes an optimal solution or its objective value, $g$ is the first committed choice, $P_g$ is the reduced instance, and <code>combine</code> attaches that choice to the optimal residual answer. The equation expresses a dependency, not a proof: dynamic programming also relies on optimal substructure. What distinguishes greedy algorithms is the additional proof that one particular <code>g</code> can be selected without comparing all possible first choices.

The generic control flow is short:

~~~text
GREEDY-SOLVE(problem P)
    solution <- empty
    while P is not complete
        candidates <- feasible choices in P
        g <- candidate preferred by the greedy rule
        add g to solution
        P <- residual problem after committing to g
    return solution
~~~

![A safe greedy choice reduces the problem to one residual subproblem, whereas dynamic programming may retain several states.](assets/greedy-choice-optimal-substructure.svg){fig-align="center" width="100%"}

**Example: fractional knapsack.** Each item $i$ has value $v_i$, weight $w_i$, and value density

$$
\rho_i=\frac{v_i}{w_i}.
$$

$\rho_i$ is the value obtained per unit of capacity. Because an item may be split, taking available weight from the highest-density item is safe: if a proposed solution uses some lower-density material while higher-density material remains available, exchanging equal weight between them cannot reduce feasibility and strictly increases value. After the exchange, the unused capacity is simply a smaller fractional-knapsack instance. Both required properties are present.

This proof depends on **fractionality**. In 0/1 knapsack, an item is indivisible, so exchanging equal amounts may be impossible; the same ratio rule can fail.

<details>
<summary>Python implementation: fractional knapsack</summary>

~~~python
from typing import Iterable


def fractional_knapsack(
    items: Iterable[tuple[str, float, float]], capacity: float
) -> tuple[float, list[tuple[str, float]]]:
    """Return maximum value and the fraction taken from each chosen item."""
    if capacity < 0:
        raise ValueError("capacity must be non-negative")

    ranked = []
    for name, value, weight in items:
        if weight <= 0:
            raise ValueError("every item weight must be positive")
        ranked.append((value / weight, name, float(value), float(weight)))

    # The greedy rule only needs this order; no later decision revisits it.
    ranked.sort(key=lambda item: item[0], reverse=True)

    total_value = 0.0
    remaining = float(capacity)
    taken: list[tuple[str, float]] = []

    for density, name, value, weight in ranked:
        if remaining == 0:
            break

        # A whole item is preferred, but the final item may be fractional.
        fraction = min(1.0, remaining / weight)
        total_value += fraction * value
        remaining -= fraction * weight
        taken.append((name, fraction))

    return total_value, taken


example_items = [
    ("A", 60, 10),
    ("B", 100, 20),
    ("C", 120, 30),
]
value, selection = fractional_knapsack(example_items, capacity=50)
assert abs(value - 240.0) < 1e-9
print(value, selection)
~~~

</details>

Sorting $n$ items costs $O(n\log n)$ time, and the scan costs $O(n)$. The returned selection occupies $O(n)$ space; excluding that output, the scan uses $O(1)$ additional state after sorting.

**Practice:** [LeetCode 455 - Assign Cookies](https://leetcode.com/problems/assign-cookies/) asks you to identify a safe ordering and match choices without revisiting earlier assignments.


### **Designing a Greedy Strategy** {#designing-a-greedy-strategy}

Designing a greedy algorithm starts by making the problem operational. A vague rule such as "take the best option" is not enough: the candidate set, feasibility condition, objective, tie behavior, and maintained state must all be explicit.

A useful design sequence is:

1. **Model a partial solution.** State exactly what has already been committed and what remains undecided.
2. **Identify feasible next choices.** A choice must preserve every hard constraint, not merely improve the objective.
3. **Propose a local key.** Examples include earliest finishing time, smallest edge weight, greatest value density, or farthest reachable position.
4. **Find the frontier state.** Good greedy algorithms often reduce all prior history to a small invariant such as the last finish time, the current reachable boundary, or the connected component containing a vertex.
5. **Prove the choice is safe.** Use an exchange, stays-ahead, cut, or induction argument.
6. **Search for tiny counterexamples.** Exhaustive comparison on small instances can refute an incorrect rule before it becomes a polished implementation.

~~~text
DESIGN-AND-RUN-GREEDY(instance)
    define candidates, feasibility, and objective
    define a local ordering or priority rule
    initialize a compact state describing the current frontier

    while the solution is incomplete
        expose the best currently feasible candidate
        commit to it
        update the frontier state

    prove that every commitment is safe
    test small instances against an exact oracle
~~~

![A disciplined workflow for proposing, proving, and attacking a greedy rule.](assets/greedy-design-workflow.svg){fig-align="center" width="100%"}

**Example: minimum refills on a fixed route.** Suppose stations are ordered along a road and a vehicle can travel at most $R$ distance after each refill. From the current position, stopping at an earlier reachable station cannot help more than stopping at the farthest reachable station: the farther stop leaves every station reachable by the earlier choice still reachable, and may expose additional stations. The greedy state is only the current station index.

For the route $[0,100,200,275,375,550]$ and range $R=200$, the vehicle jumps from $0$ to $200$, then to $375$, then reaches $550$. It never scans backward.

~~~text
MINIMUM-REFILLS(points, range R)
    current <- 0
    stops <- empty list
    while current is not the destination
        next <- current
        while the following point is within R of points[current]
            advance next
        if next = current
            return impossible
        if next is not the destination
            append points[next] to stops
        current <- next
    return stops
~~~

<details>
<summary>Python implementation: farthest-reachable refill strategy</summary>

~~~python
def minimum_refill_stops(
    points: list[int], max_distance: int
) -> list[int] | None:
    """Return refill positions, or None when the destination is unreachable."""
    if not points or points[0] != 0:
        raise ValueError("points must start at position 0")
    if max_distance <= 0:
        raise ValueError("max_distance must be positive")
    if any(points[i] >= points[i + 1] for i in range(len(points) - 1)):
        raise ValueError("points must be strictly increasing")

    current = 0
    stops: list[int] = []

    while current < len(points) - 1:
        farthest = current

        # Advance one shared pointer to the farthest point reachable now.
        while (
            farthest + 1 < len(points)
            and points[farthest + 1] - points[current] <= max_distance
        ):
            farthest += 1

        if farthest == current:
            return None  # The next gap is longer than the vehicle's range.

        if farthest < len(points) - 1:
            stops.append(points[farthest])
        current = farthest

    return stops


route = [0, 100, 200, 275, 375, 550]
assert minimum_refill_stops(route, 200) == [200, 375]
assert minimum_refill_stops([0, 150, 400], 200) is None
print(minimum_refill_stops(route, 200))
~~~

</details>

The scan is $O(n)$ time because the farthest pointer only moves forward across $n$ route points. The returned list uses $O(n)$ space in the worst case; the algorithm itself keeps $O(1)$ auxiliary state. This linear bound is produced by the monotone frontier, not by the word "greedy."

**Practice:** [LeetCode 55 - Jump Game](https://leetcode.com/problems/jump-game/) is a good exercise in replacing many possible paths with one farthest-reachable frontier.


### **Exchange Arguments and Stays-Ahead Proofs** {#exchange-arguments-and-stays-ahead-proofs}

A greedy algorithm usually has obvious feasibility and non-obvious optimality. Two proof templates cover many of the non-obvious cases.

An **exchange argument** begins with an arbitrary optimal solution $O$. If its first choice differs from the greedy choice $g$, replace part of $O$ with $g$ and show that the modified solution $O'$ remains feasible and is no worse. Therefore an optimal solution consistent with the greedy choice exists. The same reasoning is then applied to the residual problem.

A **stays-ahead proof** compares prefixes. Let $G_i$ describe the progress made after the first $i$ greedy choices and $O_i$ the progress after the first $i$ choices of any optimal solution. Prove an inequality such as $G_i\ge O_i$ for a maximization frontier or $G_i\le O_i$ for a cost or finish-time frontier, for every $i$. If greedy is never behind at any prefix, the final solution cannot be worse.

~~~text
EXCHANGE-PROOF(greedy solution G, optimal solution O)
    find the first position where G and O differ
    replace O's choice at that position with G's choice
    prove feasibility is preserved
    prove the objective value does not worsen
    repeat or invoke induction on the residual problem

STAYS-AHEAD-PROOF(G, O)
    define one comparable prefix measure M
    prove the base case M(G_1) is at least as good as M(O_1)
    assume the claim for prefix i - 1
    prove it for prefix i
    conclude the final greedy objective is optimal
~~~

![Exchange arguments transform an optimal solution; stays-ahead proofs compare every prefix.](assets/greedy-proof-patterns.svg){fig-align="center" width="100%"}

**Example: pairing loads under a two-person capacity.** Sort all weights. Consider the heaviest remaining person $h$. If $h$ cannot share with the lightest remaining person $l$, then $h$ cannot share with anyone and must travel alone. If $h+l$ fits, pairing $h$ with $l$ is safe: any solution pairing $h$ with a heavier partner can exchange that partner for $l$, preserving feasibility and not increasing the number of boats. Each step removes one unavoidable heaviest person and possibly one lightest partner.

<details>
<summary>Python implementation: exchange-safe two-person pairing</summary>

~~~python
def minimum_two_person_boats(weights: list[int], limit: int) -> int:
    """Return the minimum boats when each boat holds at most two people."""
    if limit <= 0 or any(weight <= 0 for weight in weights):
        raise ValueError("weights and limit must be positive")
    if any(weight > limit for weight in weights):
        raise ValueError("every person must fit individually")

    ordered = sorted(weights)
    lightest = 0
    heaviest = len(ordered) - 1
    boats = 0

    while lightest <= heaviest:
        # The heaviest remaining person must use one boat in every solution.
        if lightest < heaviest and ordered[lightest] + ordered[heaviest] <= limit:
            lightest += 1  # Use otherwise-wasted capacity with the lightest.
        heaviest -= 1
        boats += 1

    return boats


assert minimum_two_person_boats([3, 2, 2, 1], limit=3) == 3
assert minimum_two_person_boats([1, 2], limit=3) == 1
print(minimum_two_person_boats([3, 2, 2, 1], limit=3))
~~~

</details>

Sorting costs $O(n\log n)$ time. The two pointers then move inward at most $n$ times, so the scan is $O(n)$. Python's sorted copy occupies $O(n)$ space; an in-place sort can reduce algorithm-specific auxiliary state.

The proof should mirror the code. The code always removes the heaviest item; the argument explains why every solution must allocate capacity to it and why the lightest feasible partner is exchange-safe. A proof about a different rule would not establish this implementation's correctness.

**Practice:** [LeetCode 881 - Boats to Save People](https://leetcode.com/problems/boats-to-save-people/) directly exercises this exchange argument.


### **Interval Scheduling and Interval Merging** {#interval-scheduling-and-interval-merging}

Intervals look similar in memory but support different objectives. **Interval scheduling** chooses the largest possible subset of mutually compatible intervals. **Interval merging** computes the union of all covered ranges. Confusing the two leads to the wrong sorting key and invariant.

Assume half-open intervals $[s_i,f_i)$, where $s_i$ is the start and $f_i$ is the finish. Two adjacent intervals are compatible when the next start satisfies $s_j\ge f_i$. For closed intervals or application-specific endpoint rules, this predicate must be adjusted explicitly.

For maximum-cardinality scheduling, sort by increasing finish time and accept the next compatible interval. The earliest finish releases the shared resource as soon as possible, leaving at least as much room for all later intervals as any alternative first choice.

~~~text
INTERVAL-SCHEDULE(intervals)
    sort intervals by increasing finish time
    selected <- empty list
    last_finish <- negative infinity
    for interval (start, finish) in sorted order
        if start >= last_finish
            append interval to selected
            last_finish <- finish
    return selected

MERGE-INTERVALS(intervals)
    sort intervals by increasing start time
    merged <- empty list
    for interval (start, finish) in sorted order
        if merged is empty or start > merged.last.finish
            append a new interval
        else
            merged.last.finish <- max(merged.last.finish, finish)
    return merged
~~~

![Earliest finish creates a scheduling frontier; earliest start creates a merging frontier.](assets/interval-greedy-steps.svg){fig-align="center" width="100%"}

The scheduling proof is an exchange argument. Let $g$ be the interval with earliest finish and let $o$ be the first interval in an optimal schedule. Replacing $o$ with $g$ cannot make any later selected interval incompatible because $f_g\le f_o$. Thus an optimal schedule beginning with $g$ exists, and the intervals starting after $f_g$ form the residual instance.

Merging has a different invariant: after processing the first $i$ sorted intervals, <code>merged</code> exactly represents their union, and its final interval is the only one that may still overlap a future input. Sorting by start guarantees that no future interval can reach backward past an already finalized gap.

<details>
<summary>Python implementations: scheduling and merging</summary>

~~~python
from math import inf


def maximum_compatible_intervals(
    intervals: list[tuple[int, int]],
) -> list[tuple[int, int]]:
    """Select a maximum-cardinality set of half-open intervals."""
    if any(start > finish for start, finish in intervals):
        raise ValueError("every interval must satisfy start <= finish")

    selected: list[tuple[int, int]] = []
    last_finish = -inf

    # Earliest finish is the safe choice for maximum cardinality.
    for start, finish in sorted(intervals, key=lambda interval: (interval[1], interval[0])):
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish

    return selected


def merge_intervals(
    intervals: list[tuple[int, int]],
) -> list[tuple[int, int]]:
    """Return the union of closed intervals as disjoint sorted intervals."""
    if any(start > finish for start, finish in intervals):
        raise ValueError("every interval must satisfy start <= finish")
    if not intervals:
        return []

    ordered = sorted(intervals)
    merged: list[list[int]] = [[ordered[0][0], ordered[0][1]]]

    for start, finish in ordered[1:]:
        if start > merged[-1][1]:
            merged.append([start, finish])  # A real gap finalizes the old range.
        else:
            merged[-1][1] = max(merged[-1][1], finish)  # Extend the frontier.

    return [(start, finish) for start, finish in merged]


jobs = [(1, 4), (3, 5), (0, 6), (5, 7), (5, 9), (8, 9)]
assert maximum_compatible_intervals(jobs) == [(1, 4), (5, 7), (8, 9)]
assert merge_intervals([(1, 3), (2, 6), (8, 10), (9, 12)]) == [(1, 6), (8, 12)]
print(maximum_compatible_intervals(jobs))
print(merge_intervals([(1, 3), (2, 6), (8, 10), (9, 12)]))
~~~

</details>

Both algorithms are dominated by sorting, so they take $O(n\log n)$ time followed by an $O(n)$ scan. Their output can contain $O(n)$ intervals. The scheduling scan stores only one frontier value beyond its output; the merge scan mutates only the final output interval because sorting makes all earlier output immutable.

| Goal | Sort key | Commit condition | Maintained frontier |
|---|---|---|---|
| Maximum number of compatible intervals | finish time | next start $\ge$ last finish | last selected finish |
| Union of covered ranges | start time | new range only after a gap | end of current merged range |

**Practice:** [LeetCode 435 - Non-overlapping Intervals](https://leetcode.com/problems/non-overlapping-intervals/) tests whether you can translate earliest-finish scheduling into the equivalent minimum-removal objective.


### **Huffman Coding** {#huffman-coding}

**Huffman coding** builds a variable-length binary code in which frequent symbols receive short codewords and rare symbols receive longer ones. The code is **prefix-free**: no symbol's codeword is a prefix of another symbol's codeword. Consequently, a decoder can read bits from left to right and emit a symbol whenever it reaches a leaf, without separators between codewords.

If symbol $i$ occurs with probability $p_i$ and receives a codeword of length $\ell_i$, the expected number of bits per symbol is

$$
L=\sum_{i=1}^{k} p_i\ell_i.
$$

$k$ is the number of distinct symbols, $p_i$ is the relative frequency of symbol $i$, $\ell_i$ is its leaf depth in the binary code tree, and $L$ is the average encoded length. Huffman's objective is to minimize this weighted external path length among binary prefix codes.

The greedy choice repeatedly combines the two least frequent roots. A min-priority queue implements the needed operations: insert each one-node tree, extract the two minimum-frequency trees, join them under a new parent whose frequency is their sum, and insert that parent again.

~~~text
HUFFMAN(frequencies)
    Q <- min-priority queue containing one leaf per symbol
    while size(Q) > 1
        left <- EXTRACT-MIN(Q)
        right <- EXTRACT-MIN(Q)
        parent.frequency <- left.frequency + right.frequency
        parent.left <- left
        parent.right <- right
        INSERT(Q, parent)
    return EXTRACT-MIN(Q) as the code tree

ASSIGN-CODES(node, prefix)
    if node is a leaf
        code[node.symbol] <- prefix
    else
        ASSIGN-CODES(node.left, prefix + "0")
        ASSIGN-CODES(node.right, prefix + "1")
~~~

![Huffman coding repeatedly merges the two least frequent entries, builds a tree, derives a dictionary, and encodes the message.](assets/huffman-coding-visualisation.svg){fig-align="center" width="90%"}

*Visual source: [Cmglee, Huffman coding visualisation](https://commons.wikimedia.org/wiki/File:Huffman_coding_visualisation.svg), CC BY-SA 4.0. The unchanged SVG is stored locally for reliable rendering.*

The correctness proof combines exchange and induction. In some optimal full binary code tree, the two least frequent symbols can be placed as deepest sibling leaves: exchanging a less frequent symbol downward and a more frequent symbol upward cannot increase $\sum p_i\ell_i$. Merge those siblings into one pseudo-symbol with combined frequency. An optimal code for the smaller alphabet expands back into an optimal code for the original alphabet. Repeating this argument justifies every priority-queue merge.

<details>
<summary>Python implementation: build, encode, and decode a Huffman code</summary>

~~~python
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from heapq import heapify, heappop, heappush
from itertools import count


@dataclass
class HuffmanNode:
    symbol: str | None = None
    left: HuffmanNode | None = None
    right: HuffmanNode | None = None


def build_huffman_codes(text: str) -> dict[str, str]:
    """Build a deterministic prefix-code dictionary for non-empty text."""
    if not text:
        raise ValueError("text must be non-empty")

    serial = count()  # Break equal-frequency ties without comparing tree nodes.
    heap = [
        (frequency, next(serial), HuffmanNode(symbol=symbol))
        for symbol, frequency in Counter(text).items()
    ]
    heapify(heap)

    if len(heap) == 1:
        only_symbol = heap[0][2].symbol
        return {only_symbol: "0"}

    while len(heap) > 1:
        left_frequency, _, left = heappop(heap)
        right_frequency, _, right = heappop(heap)

        # This parent represents the two least frequent remaining subtrees.
        parent = HuffmanNode(left=left, right=right)
        heappush(
            heap,
            (left_frequency + right_frequency, next(serial), parent),
        )

    root = heap[0][2]
    codes: dict[str, str] = {}

    def visit(node: HuffmanNode, prefix: str) -> None:
        if node.symbol is not None:
            codes[node.symbol] = prefix
            return
        visit(node.left, prefix + "0")
        visit(node.right, prefix + "1")

    visit(root, "")
    return codes


def huffman_encode(text: str, codes: dict[str, str]) -> str:
    return "".join(codes[symbol] for symbol in text)


def huffman_decode(bits: str, codes: dict[str, str]) -> str:
    reverse = {code: symbol for symbol, code in codes.items()}
    decoded: list[str] = []
    prefix = ""

    for bit in bits:
        if bit not in "01":
            raise ValueError("encoded data must contain only 0 and 1")
        prefix += bit
        if prefix in reverse:
            decoded.append(reverse[prefix])
            prefix = ""

    if prefix:
        raise ValueError("bit string ends inside a codeword")
    return "".join(decoded)


message = "beep boop beer!"
dictionary = build_huffman_codes(message)
encoded = huffman_encode(message, dictionary)
assert huffman_decode(encoded, dictionary) == message
assert all(
    not b.startswith(a)
    for a in dictionary.values()
    for b in dictionary.values()
    if a != b
)
print(dictionary)
print(f"{len(message) * 8} raw bits -> {len(encoded)} encoded bits")
~~~

</details>

For $k$ distinct symbols, heap construction is $O(k)$ and the $k-1$ merge steps cost $O(k\log k)$. Traversing the tree is $O(k)$, and encoding a message of $m$ symbols is $O(m)$ dictionary lookups. The tree, heap, and code dictionary each occupy $O(k)$ space. The encoded bit string itself is output space.

**Practice:** [LeetCode 1167 - Minimum Cost to Connect Sticks](https://leetcode.com/problems/minimum-cost-to-connect-sticks/) isolates the same greedy rule: repeatedly merge the two smallest weights because every early merge cost is paid again by later merges.


### **Minimum Spanning Trees as Greedy Algorithms** {#minimum-spanning-trees-as-greedy-algorithms}

For a connected, undirected, weighted graph $G=(V,E)$, a **minimum spanning tree** (MST) connects all vertices, contains no cycle, uses exactly $|V|-1$ edges, and minimizes total edge weight. Kruskal's and Prim's algorithms differ operationally, but both are greedy because each repeatedly commits to the lightest edge certified as safe by a cut.

A **cut** partitions the vertices into nonempty sets $S$ and $V\setminus S$. An edge crosses the cut when one endpoint lies in each set. The cut property states that a minimum-weight edge crossing a cut that respects the already selected forest is safe: at least one MST contains the selected edges together with that edge.

~~~text
LIGHTEST-CROSSING-EDGE(vertex set S, edges E)
    best <- none
    for each edge (u, v, weight) in E
        if exactly one of u and v belongs to S
            if best is none or weight < best.weight
                best <- (u, v, weight)
    return best

GREEDY-MST-SCHEME(graph)
    A <- empty set of selected edges
    while A has fewer than |V| - 1 edges
        choose a cut that respects A
        add a lightest edge crossing that cut
    return A
~~~

![The cut property: the lightest edge crossing a valid cut can be added safely to an MST.](assets/mst-cut-property.svg){fig-align="center" width="78%"}

*Visual source: [Ftiercel, Cut and minimum spanning tree of a graph](https://commons.wikimedia.org/wiki/File:Cut_and_minimum_spanning_tree_of_a_graph.svg), CC BY-SA 3.0. The unchanged SVG is stored locally for reliable rendering.*

Why is the crossing edge safe? Let $e$ be the selected light edge and let $T$ be an MST that does not contain it. Adding $e$ to $T$ creates one cycle. That cycle must contain another edge $f$ crossing the same cut. Since $e$ is lightest, $w(e)\le w(f)$. Replacing $f$ with $e$ restores a spanning tree and does not increase total weight. This exchange creates an MST containing $e$.

- **Kruskal** sorts all edges and uses Disjoint Set Union to reject edges whose endpoints are already connected. Its implicit cut separates one current component from the others.
- **Prim** grows one connected tree and uses a priority queue to choose the lightest edge from the tree to an outside vertex. Its cut is the current tree versus all unvisited vertices.

The complete data structures, step diagrams, and implementations are in [Minimum Spanning Trees in Chapter 05](05-graphs-and-core-algorithms.html#minimum-spanning-trees). This section focuses on the common greedy proof rather than repeating them.

<details>
<summary>Python implementation: expose the safe edge across a cut</summary>

~~~python
from typing import Hashable


def lightest_crossing_edge(
    left_side: set[Hashable],
    edges: list[tuple[Hashable, Hashable, float]],
) -> tuple[Hashable, Hashable, float] | None:
    """Return a lightest edge with exactly one endpoint in left_side."""
    best: tuple[Hashable, Hashable, float] | None = None

    for u, v, weight in edges:
        crosses = (u in left_side) != (v in left_side)  # Logical XOR.
        if crosses and (best is None or weight < best[2]):
            best = (u, v, weight)

    return best


weighted_edges = [
    ("A", "B", 4),
    ("A", "C", 2),
    ("B", "C", 1),
    ("B", "D", 5),
    ("C", "D", 8),
]
cut = {"A", "C"}
assert lightest_crossing_edge(cut, weighted_edges) == ("B", "C", 1)
print(lightest_crossing_edge(cut, weighted_edges))
~~~

</details>

The cut scan takes $O(|E|)$ time and $O(1)$ auxiliary space because it retains only the best crossing edge. A full Kruskal implementation takes $O(|E|\log |E|)$ time for sorting plus near-constant amortized DSU operations. Prim with adjacency lists and a binary heap takes $O((|V|+|E|)\log |V|)$ time and $O(|V|+|E|)$ graph-and-heap space.

**Practice:** [LeetCode 1584 - Min Cost to Connect All Points](https://leetcode.com/problems/min-cost-to-connect-all-points/) asks you to turn geometric connection costs into an MST and choose either Prim or Kruskal.


### **When Greedy Algorithms Fail** {#when-greedy-algorithms-fail}

A greedy rule fails when a locally preferred commitment can destroy a globally better combination. Optimal substructure alone does not rescue it: coin change, 0/1 knapsack, and weighted interval scheduling all admit optimal subproblems, yet natural one-path rules can choose the wrong residual problem.

Typical warning signs include:

| Warning sign | Why it threatens greedy reasoning | Likely alternative |
|---|---|---|
| A choice consumes an indivisible resource | Equal-size exchanges may be impossible | dynamic programming |
| The value of an item depends on later combinations | A local score omits interaction effects | DP or search |
| A better prefix can produce a worse completion | No stays-ahead invariant exists | retain multiple states |
| Negative weights reverse an earlier comparison | A finalized choice may need revision | Bellman-Ford-style relaxation |
| The proposed proof only gives an example | Examples establish plausibility, not universal safety | exchange proof or counterexample search |

To attack a rule, construct the smallest input on which its first choice differs from an exact optimum. The following two algorithms expose the coin-change failure:

~~~text
GREEDY-COINS(coins, amount)
    sort coins from largest to smallest
    repeatedly take as many as possible of the current coin

OPTIMAL-COINS(coins, amount)
    dp[0] <- 0
    for value from 1 to amount
        dp[value] <- 1 + minimum dp[value - coin] over feasible coins
    return dp[amount]
~~~

![Two natural greedy rules fail because their first commitments block better combinations.](assets/greedy-counterexamples.svg){fig-align="center" width="100%"}

With coins $\{1,3,4\}$ and target $6$, largest-first chooses $4+1+1$ using three coins, while the optimum is $3+3$ using two. In 0/1 knapsack, density order works for fractional material but can waste indivisible capacity. Similarly, earliest finish maximizes the **number** of compatible intervals, but it does not maximize arbitrary interval weights; weighted interval scheduling needs states that compare take and skip choices.

<details>
<summary>Python implementation: compare a greedy rule with an exact DP oracle</summary>

~~~python
from math import inf


def greedy_coin_count(coins: list[int], amount: int) -> int | None:
    """Use the largest-first rule; this is not correct for every coin system."""
    if amount < 0 or any(coin <= 0 for coin in coins):
        raise ValueError("amount and coin values must be non-negative/positive")

    count = 0
    remaining = amount
    for coin in sorted(set(coins), reverse=True):
        used, remaining = divmod(remaining, coin)
        count += used
    return count if remaining == 0 else None


def optimal_coin_count(coins: list[int], amount: int) -> int | None:
    """Use dynamic programming as an exact oracle for minimum coin count."""
    if amount < 0 or any(coin <= 0 for coin in coins):
        raise ValueError("amount and coin values must be non-negative/positive")

    dp = [0] + [inf] * amount
    for value in range(1, amount + 1):
        for coin in coins:
            if coin <= value:
                dp[value] = min(dp[value], 1 + dp[value - coin])
    return None if dp[amount] == inf else int(dp[amount])


coins = [1, 3, 4]
assert greedy_coin_count(coins, 6) == 3
assert optimal_coin_count(coins, 6) == 2
print("greedy:", greedy_coin_count(coins, 6))
print("optimal:", optimal_coin_count(coins, 6))
~~~

</details>

For $k$ denominations, the greedy method takes $O(k\log k)$ time to sort and $O(1)$ extra state. The exact DP oracle takes $O(kA)$ time and $O(A)$ space for target amount $A$. That additional state is precisely what allows DP to preserve alternatives that the greedy rule discards.

**Practice:** [LeetCode 322 - Coin Change](https://leetcode.com/problems/coin-change/) is a direct reminder that a familiar local rule must not be generalized beyond the coin systems for which it has been proved.


### **Comparison and Selection** {#comparison-and-selection}

Greedy algorithms are most attractive when a small frontier summarizes all relevant history and a local choice can be proved safe. They should be selected from the problem's structure, not merely because sorting appears in a plausible solution.

| Technique | Decisions retained | Typical evidence of correctness | Typical cost profile | Use when |
|---|---:|---|---|---|
| Greedy | one committed path | exchange, stays-ahead, cut property | often $O(n)$ or $O(n\log n)$ | one safe local choice dominates alternatives |
| Dynamic programming | many summarized states | recurrence and induction | state count times transition cost | choices interact and subproblems overlap |
| Divide-and-conquer | independent recursive branches | decomposition plus correct combine step | recurrence such as $T(n)=aT(n/b)+f(n)$ | subproblems can be solved independently |
| Backtracking | explicit alternatives in a state-space tree | exhaustive coverage plus pruning safety | often exponential worst case | constraints require enumerating compatible combinations |

Before committing to greedy, ask:

1. **What is the exact objective?** Maximum count, maximum weight, minimum cost, and earliest completion can require different rules on the same objects.
2. **What makes the next choice safe?** Name the exchange, prefix inequality, cut, or other structural fact.
3. **What residual problem remains?** It should preserve the form required by the proof.
4. **Which state is truly sufficient?** If one frontier value cannot summarize the past, multiple DP states may be necessary.
5. **Can a tiny exact solver disagree?** Differential testing is excellent for discovering counterexamples, even though testing cannot replace a universal proof.

<details>
<summary>Python implementation: find the first counterexample to a greedy hypothesis</summary>

~~~python
from collections.abc import Callable, Iterable
from math import inf
from typing import TypeVar

T = TypeVar("T")


def first_counterexample(
    instances: Iterable[T],
    candidate: Callable[[T], int | None],
    oracle: Callable[[T], int | None],
) -> tuple[T, int | None, int | None] | None:
    """Return the first small instance on which candidate and oracle disagree."""
    for instance in instances:
        candidate_value = candidate(instance)
        oracle_value = oracle(instance)
        if candidate_value != oracle_value:
            return instance, candidate_value, oracle_value
    return None


coins = (1, 3, 4)


def largest_first(amount: int) -> int:
    count = 0
    for coin in sorted(coins, reverse=True):
        used, amount = divmod(amount, coin)
        count += used
    return count


def exact_minimum(amount: int) -> int:
    dp = [0] + [inf] * amount
    for value in range(1, amount + 1):
        dp[value] = 1 + min(dp[value - coin] for coin in coins if coin <= value)
    return int(dp[amount])


failure = first_counterexample(range(1, 20), largest_first, exact_minimum)
assert failure == (6, 3, 2)
print(failure)
~~~

</details>

The harness makes one candidate and one oracle call per test instance. Its total cost is therefore the number of tested instances multiplied by the cost of those two solvers. Use it only on deliberately small inputs when the oracle is expensive. Once a counterexample appears, inspect the first divergent commitment and revise either the rule or the algorithmic paradigm.

**Practice:** [LeetCode 45 - Jump Game II](https://leetcode.com/problems/jump-game-ii/) is a useful synthesis exercise: the correct greedy state is a level-like reachable frontier, not a commitment to one literal jump path.
